<a href="https://colab.research.google.com/github/AkhilanairK-healthcare/akhila-hospital-readmission-quality/blob/main/notebooks/01_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
BASE = '/content/drive/MyDrive/Akhila-portfolio/Project2-data/'
hrrp = pd.read_csv(BASE + 'hrrp.csv', dtype={'Facility ID': str})
hosp = pd.read_csv(BASE + 'hospital_general.csv', dtype={'Facility ID': str})
print(hrrp.shape, hosp.shape)
print(hrrp.columns.tolist())

Mounted at /content/drive
(18330, 12) (5435, 38)
['Facility Name', 'Facility ID', 'State', 'Measure Name', 'Number of Discharges', 'Footnote', 'Excess Readmission Ratio', 'Predicted Readmission Rate', 'Expected Readmission Rate', 'Number of Readmissions', 'Start Date', 'End Date']


In [5]:
def clean_cols(df):
    df.columns = (df.columns.str.strip().str.lower()
                  .str.replace(' ', '_').str.replace('/', '_'))
    return df
hrrp = clean_cols(hrrp)
hosp = clean_cols(hosp)
print(hrrp.columns.tolist())

['facility_name', 'facility_id', 'state', 'measure_name', 'number_of_discharges', 'footnote', 'excess_readmission_ratio', 'predicted_readmission_rate', 'expected_readmission_rate', 'number_of_readmissions', 'start_date', 'end_date']


In [6]:
bad = ['Not Available', 'N/A', 'Too Few to Report', '']
num_cols = ['number_of_discharges', 'excess_readmission_ratio',
  'predicted_readmission_rate', 'expected_readmission_rate',
  'number_of_readmissions']
for c in num_cols:
    hrrp[c] = pd.to_numeric(hrrp[c].replace(bad, pd.NA), errors='coerce')
print(hrrp[num_cols].isna().sum())

number_of_discharges          10088
excess_readmission_ratio       6610
predicted_readmission_rate     6610
expected_readmission_rate      6610
number_of_readmissions        10293
dtype: int64


In [7]:
hosp = hosp[['facility_id','facility_name','state',
  'hospital_type','hospital_ownership','hospital_overall_rating']].copy()
hosp['hospital_overall_rating'] = pd.to_numeric(
    hosp['hospital_overall_rating'].replace(bad, pd.NA), errors='coerce')
hosp = hosp.drop_duplicates(subset=['facility_id'])
print('hospitals:', len(hosp))

hospitals: 5435


In [8]:
names = {'READM-30-HF-HRRP': 'Heart Failure',
 'READM-30-PN-HRRP': 'Pneumonia',
 'READM-30-AMI-HRRP': 'Heart Attack (AMI)',
 'READM-30-COPD-HRRP': 'COPD',
 'READM-30-HIP-KNEE-HRRP': 'Hip/Knee Replacement',
 'READM-30-CABG-HRRP': 'CABG (Bypass Surgery)'}
hrrp['condition'] = hrrp['measure_name'].map(names)
hrrp['above_benchmark'] = (hrrp['excess_readmission_ratio'] > 1.0).astype('Int64')
hrrp.loc[hrrp['excess_readmission_ratio'].isna(), 'above_benchmark'] = pd.NA
print(hrrp['condition'].value_counts())

condition
Hip/Knee Replacement     3055
CABG (Bypass Surgery)    3055
Heart Attack (AMI)       3055
COPD                     3055
Pneumonia                3055
Heart Failure            3055
Name: count, dtype: int64


In [9]:
import sqlite3
merged = hrrp.merge(hosp, on=['facility_id','facility_name','state'],
                    how='left', indicator=True)
print(merged['_merge'].value_counts())
conn = sqlite3.connect(BASE + 'readmissions.db')
hrrp.to_sql('fact_readmissions', conn, if_exists='replace', index=False)
hosp.to_sql('dim_hospital', conn, if_exists='replace', index=False)
conn.close()
hrrp.to_csv(BASE + 'fact_readmissions.csv', index=False)
hosp.to_csv(BASE + 'dim_hospital.csv', index=False)
print('saved')

_merge
both          17610
left_only       720
right_only        0
Name: count, dtype: int64
saved


In [10]:
merged2 = hrrp.merge(hosp, on='facility_id', how='left', indicator=True)
print(merged2['_merge'].value_counts())

_merge
both          18108
left_only       222
right_only        0
Name: count, dtype: int64


In [11]:
print(hrrp.groupby('condition')['excess_readmission_ratio'].mean().round(4))
print(hrrp.groupby('condition')['above_benchmark'].mean().round(4) * 100)

condition
CABG (Bypass Surgery)    1.0018
COPD                     1.0011
Heart Attack (AMI)       1.0018
Heart Failure            1.0014
Hip/Knee Replacement     1.0040
Pneumonia                1.0015
Name: excess_readmission_ratio, dtype: float64
condition
CABG (Bypass Surgery)    49.89
COPD                     47.22
Heart Attack (AMI)       49.65
Heart Failure            48.91
Hip/Knee Replacement     47.89
Pneumonia                46.81
Name: above_benchmark, dtype: Float64


In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
import pandas as pd
import sqlite3
BASE = '/content/drive/MyDrive/Akhila-portfolio/Project2-data/'
conn = sqlite3.connect(BASE + 'readmissions.db')
def q(sql):
    return pd.read_sql_query(sql, conn)
print('connected')

connected


In [15]:
q("""SELECT condition, COUNT(excess_readmission_ratio) AS hospitals_with_data,
  ROUND(AVG(excess_readmission_ratio), 4) AS avg_err
FROM fact_readmissions GROUP BY condition ORDER BY avg_err DESC;""")

,condition,hospitals_with_data,avg_err
0,Hip/Knee Replacement,1447,1.0040
1,Heart Attack (AMI),1736,1.0018
2,CABG (Bypass Surgery),878,1.0018
3,Pneumonia,2715,1.0015
4,Heart Failure,2621,1.0014
5,COPD,2323,1.0011


In [17]:
q("""SELECT condition, COUNT(excess_readmission_ratio) AS hospitals_with_data,
  ROUND(100.0 * AVG(above_benchmark), 1) AS pct_above_benchmark
FROM fact_readmissions GROUP BY condition ORDER BY pct_above_benchmark DESC;""")

,condition,hospitals_with_data,pct_above_benchmark
0,CABG (Bypass Surgery),878,49.9
1,Heart Attack (AMI),1736,49.7
2,Heart Failure,2621,48.9
3,Hip/Knee Replacement,1447,47.9
4,COPD,2323,47.2
5,Pneumonia,2715,46.8


In [18]:
q("""SELECT facility_name, state, number_of_discharges,
  excess_readmission_ratio, number_of_readmissions
FROM fact_readmissions
WHERE condition = 'Heart Failure' AND excess_readmission_ratio IS NOT NULL
ORDER BY excess_readmission_ratio DESC LIMIT 15;""")

,facility_name,state,number_of_discharges,excess_readmission_ratio,number_of_readmissions
0,OROVILLE HOSPITAL,CA,285.0,1.3606,98.0
1,ST LUCIE MEDICAL CENTER,FL,390.0,1.2781,115.0
2,SOIN MEDICAL CENTER,OH,364.0,1.2443,110.0
3,"NEW YORK COMMUNITY HOSPITAL OF BROOKLYN, INC.",NY,409.0,1.2419,120.0
4,BROWARD HEALTH NORTH,FL,194.0,1.2408,62.0
5,WEST CHESTER HOSPITAL,OH,348.0,1.2312,99.0
6,WINCHESTER HOSPITAL,MA,685.0,1.2300,172.0
7,CENTRAL PENINSULA GENERAL HOSPITAL,AK,181.0,1.2271,57.0
8,JEFFERSON HEALTH- NORTHEAST,PA,839.0,1.2197,218.0
9,MILFORD REGIONAL MEDICAL CENTER,MA,733.0,1.2135,180.0


In [19]:
q("""SELECT facility_name, state, number_of_discharges,
  excess_readmission_ratio, number_of_readmissions
FROM fact_readmissions
WHERE condition = 'Pneumonia' AND excess_readmission_ratio IS NOT NULL
ORDER BY excess_readmission_ratio DESC LIMIT 15;""")

,facility_name,state,number_of_discharges,excess_readmission_ratio,number_of_readmissions
0,OROVILLE HOSPITAL,CA,833.0,1.4255,229.0
1,ST JOHN'S EPISCOPAL HOSPITAL AT SOUTH SHORE,NY,263.0,1.3274,81.0
2,MAIMONIDES MEDICAL CENTER,NY,975.0,1.3011,250.0
3,MERITUS MEDICAL CENTER,MD,896.0,1.2406,189.0
4,MARY WASHINGTON HOSPITAL,VA,1178.0,1.2380,251.0
5,ALLIANCEHEALTH DURANT,OK,415.0,1.2349,100.0
6,SAN DIMAS COMMUNITY HOSPITAL,CA,185.0,1.2289,55.0
7,JEFFERSON STRATFORD HOSPITAL,NJ,637.0,1.2278,138.0
8,RICHMOND UNIVERSITY MEDICAL CENTER,NY,303.0,1.2130,78.0
9,REGIONAL MEDICAL CENTER OF SAN JOSE,CA,499.0,1.2125,102.0


In [20]:
q("""SELECT facility_name, state, number_of_discharges,
  excess_readmission_ratio, number_of_readmissions
FROM fact_readmissions
WHERE condition = 'Heart Attack (AMI)' AND excess_readmission_ratio IS NOT NULL
ORDER BY excess_readmission_ratio DESC LIMIT 15;""")

,facility_name,state,number_of_discharges,excess_readmission_ratio,number_of_readmissions
0,BOSTON MEDICAL CENTER-BRIGHTON,MA,329.0,1.2639,73.0
1,TUCSON MEDICAL CENTER,AZ,314.0,1.2525,56.0
2,MEDICAL CITY FORT WORTH,TX,142.0,1.2367,36.0
3,HCA FLORIDA BAYONET POINT HOSPITAL,FL,151.0,1.2334,40.0
4,GOOD SAMARITAN HOSPITAL,CA,256.0,1.2312,52.0
5,BRIGHAM AND WOMEN'S HOSPITAL,MA,660.0,1.2243,125.0
6,TUFTS MEDICAL CENTER,MA,197.0,1.2234,48.0
7,CENTINELA HOSPITAL MEDICAL CENTER,CA,86.0,1.2168,33.0
8,"NORTON HOSPITALS, INC",KY,642.0,1.1967,115.0
9,SOUTH SHORE HOSPITAL,MA,351.0,1.1911,65.0


In [21]:
q("""SELECT facility_name, state, number_of_discharges,
  excess_readmission_ratio, number_of_readmissions
FROM fact_readmissions
WHERE condition = 'COPD' AND excess_readmission_ratio IS NOT NULL
ORDER BY excess_readmission_ratio DESC LIMIT 15;""")

,facility_name,state,number_of_discharges,excess_readmission_ratio,number_of_readmissions
0,OROVILLE HOSPITAL,CA,99.0,1.2545,44.0
1,ST LUKE'S HOSPITAL - MONROE CAMPUS,PA,133.0,1.1944,41.0
2,DCH REGIONAL MEDICAL CENTER,AL,270.0,1.1728,71.0
3,HCA FLORIDA NORTHSIDE HOSPITAL,FL,127.0,1.1610,42.0
4,ST MARY MEDICAL CENTER,PA,237.0,1.1610,65.0
5,MILFORD REGIONAL MEDICAL CENTER,MA,367.0,1.1580,86.0
6,MONTEFIORE MEDICAL CENTER,NY,273.0,1.1556,79.0
7,JOHN MUIR MEDICAL CENTER - CONCORD CAMPUS,CA,124.0,1.1544,37.0
8,LEHIGH VALLEY HOSPITAL,PA,555.0,1.1530,128.0
9,GOOD SAMARITAN MEDICAL CENTER,FL,78.0,1.1479,26.0


In [22]:
q("""SELECT facility_name, state, number_of_discharges,
  excess_readmission_ratio, number_of_readmissions
FROM fact_readmissions
WHERE condition = 'Hip/Knee Replacement' AND excess_readmission_ratio IS NOT NULL
ORDER BY excess_readmission_ratio DESC LIMIT 15;""")

,facility_name,state,number_of_discharges,excess_readmission_ratio,number_of_readmissions
0,FAIRFIELD MEDICAL CENTER,OH,NaN,1.6297,NaN
1,CARLE BROMENN MEDICAL CENTER,IL,292.0,1.5827,27.0
2,HOLY FAMILY HOSPITAL,MA,131.0,1.5658,14.0
3,BROWN UNIVERSITY HEALTH MORTON HOSPITAL,MA,239.0,1.5594,21.0
4,GREAT PLAINS REGIONAL MEDICAL CENTER,OK,NaN,1.5367,NaN
5,SAINT FRANCIS HOSPITAL MUSKOGEE,OK,102.0,1.5217,12.0
6,HCA FLORIDA OAK HILL HOSPITAL,FL,138.0,1.5196,16.0
7,ST MARY'S HOSPITAL,GA,111.0,1.5095,15.0
8,GOOD SAMARITAN MEDICAL CENTER,FL,101.0,1.4908,14.0
9,ECU HEALTH MEDICAL CENTER,NC,171.0,1.4851,19.0


In [23]:
q("""SELECT facility_name, state, number_of_discharges,
  excess_readmission_ratio, number_of_readmissions
FROM fact_readmissions
WHERE condition = 'CABG (Bypass Surgery)' AND excess_readmission_ratio IS NOT NULL
ORDER BY excess_readmission_ratio DESC LIMIT 15;""")

,facility_name,state,number_of_discharges,excess_readmission_ratio,number_of_readmissions
0,ROBERT WOOD JOHNSON UNIVERSITY HOSPITAL,NJ,357.0,1.4237,68.0
1,MCLAREN BAY REGION,MI,103.0,1.3880,25.0
2,PHYSICIANS REGIONAL MEDICAL CENTER,TN,149.0,1.3264,28.0
3,JOHNSON CITY MEDICAL CENTER,TN,101.0,1.2914,19.0
4,UNIVERSITY OF ALABAMA HOSPITAL,AL,358.0,1.2863,61.0
5,CHARLESTON AREA MEDICAL CENTER,WV,298.0,1.2837,48.0
6,MERCY HOSPITAL ST LOUIS,MO,175.0,1.2812,32.0
7,HCA FLORIDA BAYONET POINT HOSPITAL,FL,85.0,1.2553,20.0
8,DOCTORS MEDICAL CENTER,CA,130.0,1.2466,22.0
9,BAYSTATE MEDICAL CENTER,MA,310.0,1.2464,49.0


In [24]:
q("""SELECT state, COUNT(excess_readmission_ratio) AS hospital_measures,
  ROUND(AVG(excess_readmission_ratio), 4) AS avg_err
FROM fact_readmissions GROUP BY state
HAVING hospital_measures >= 100 ORDER BY avg_err DESC LIMIT 15;""")

,state,hospital_measures,avg_err
0,MA,240,1.0344
1,NJ,272,1.0277
2,FL,771,1.0229
3,IL,491,1.0194
4,MS,185,1.0141
5,AL,247,1.0139
6,GA,346,1.0114
7,KY,227,1.0101
8,CA,1038,1.0101
9,OH,481,1.0085


In [25]:
q("""SELECT h.hospital_ownership, COUNT(*) AS hospital_measures,
  ROUND(AVG(f.excess_readmission_ratio), 4) AS avg_err
FROM fact_readmissions f JOIN dim_hospital h ON f.facility_id = h.facility_id
GROUP BY h.hospital_ownership ORDER BY avg_err DESC;""")

,hospital_ownership,hospital_measures,avg_err
0,Proprietary,3678,1.0171
1,Government - Federal,144,1.0062
2,Government - Local,792,1.0031
3,Government - State,270,1.0008
4,Government - Hospital District or Authority,1320,1.0000
5,Voluntary non-profit - Private,8850,0.9990
6,Voluntary non-profit - Other,1404,0.9980
7,Voluntary non-profit - Church,1176,0.9945
8,Tribal,54,0.9756
9,Physician,420,0.9577


In [26]:
q("""SELECT h.hospital_type, COUNT(*) AS hospital_measures,
  ROUND(AVG(f.excess_readmission_ratio), 4) AS avg_err
FROM fact_readmissions f JOIN dim_hospital h ON f.facility_id = h.facility_id
GROUP BY h.hospital_type ORDER BY avg_err DESC;""")

,hospital_type,hospital_measures,avg_err
0,Acute Care Hospitals,18108,1.0017


In [27]:
q("""SELECT h.hospital_overall_rating, COUNT(*) AS hospital_measures,
  ROUND(AVG(f.excess_readmission_ratio), 4) AS avg_err
FROM fact_readmissions f JOIN dim_hospital h ON f.facility_id = h.facility_id
WHERE h.hospital_overall_rating IS NOT NULL
GROUP BY h.hospital_overall_rating ORDER BY h.hospital_overall_rating;""")

,hospital_overall_rating,hospital_measures,avg_err
0,1.0,1020,1.0484
1,2.0,3402,1.0250
2,3.0,5052,1.0061
3,4.0,4656,0.9855
4,5.0,1740,0.9663


In [28]:
q("""SELECT COUNT(DISTINCT facility_id) AS hospitals_in_hrrp,
  COUNT(*) AS hospital_measure_rows,
  COUNT(excess_readmission_ratio) AS rows_with_err
FROM fact_readmissions;""")

,hospitals_in_hrrp,hospital_measure_rows,rows_with_err
0,3055,18330,11720


In [29]:
q("""SELECT condition, COUNT(*) AS rows_total,
  SUM(CASE WHEN excess_readmission_ratio IS NULL THEN 1 ELSE 0 END) AS rows_missing_err,
  ROUND(100.0 * SUM(CASE WHEN excess_readmission_ratio IS NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_missing
FROM fact_readmissions GROUP BY condition ORDER BY pct_missing DESC;""")

,condition,rows_total,rows_missing_err,pct_missing
0,CABG (Bypass Surgery),3055,2177,71.3
1,Hip/Knee Replacement,3055,1608,52.6
2,Heart Attack (AMI),3055,1319,43.2
3,COPD,3055,732,24.0
4,Heart Failure,3055,434,14.2
5,Pneumonia,3055,340,11.1


In [30]:
q("""SELECT facility_name, state, number_of_discharges, excess_readmission_ratio
FROM fact_readmissions
WHERE condition = 'Pneumonia' AND excess_readmission_ratio IS NOT NULL
  AND number_of_discharges >= 100
ORDER BY excess_readmission_ratio ASC LIMIT 15;""")

,facility_name,state,number_of_discharges,excess_readmission_ratio
0,AUGUSTA HEALTH,VA,392.0,0.7967
1,TRINITY HEALTH MUSKEGON HOSPITAL,MI,330.0,0.7976
2,KOOTENAI HEALTH,ID,413.0,0.8077
3,SALINA REGIONAL HEALTH CENTER,KS,317.0,0.8166
4,CHRISTUS ST VINCENT REGIONAL MEDICAL CENTER,NM,480.0,0.8174
5,HOLMES REGIONAL MEDICAL CENTER,FL,710.0,0.8175
6,MERCY MEDICAL CENTER,OR,216.0,0.8178
7,NORTHWELL HOSPITAL GLEN COVE,NY,234.0,0.8207
8,LOGAN HEALTH MEDICAL CENTER,MT,252.0,0.8245
9,PROVIDENCE ST JOSEPH HOSPITAL,CA,431.0,0.8248
